# 10 — Building Your Own Eval Set

Every notebook so far borrowed its questions from someone else's benchmark -- CUAD, SQuAD2, HotpotQA, MS MARCO. That's honest for learning (zero authoring, real ground truth already attached), but it skips the single most common real-world task: you have *your own* documents, nobody has written test questions for them, and you need a benchmark before you can measure anything.

The theory phase covered this idea in the abstract. This notebook actually does it: generate candidate question-answer pairs from a real document, then filter them for quality before trusting any of them.


In [ ]:
import os
import json
from dotenv import load_dotenv
from groq import Groq
from sentence_transformers import SentenceTransformer

load_dotenv()

client = Groq(api_key=os.environ["GROQ_API_KEY"])
MODEL = "openai/gpt-oss-20b"
embedder = SentenceTransformer("all-MiniLM-L6-v2")

DATA_DIR = os.path.join("..", "data")


## A real source document

Same CUAD contract this project has already touched once before (notebook 04's "Cap On Liability" example came from the same company) -- a full web hosting agreement, not a trimmed 700-character window this time. Real length, real structure, real clauses nobody labeled for you.


In [ ]:
from datasets import load_dataset

cuad = load_dataset("chenghao/cuad_qa", split="test", streaming=True)


def first_n_matching(dataset, predicate, n, scan_limit=5000):
    out = []
    for i, ex in enumerate(dataset):
        if predicate(ex):
            out.append(ex)
        if len(out) == n or i >= scan_limit:
            break
    return out


# The second "Governing Law" match in this dataset is the same i-on web hosting agreement
# notebook 04 pulled a liability clause from -- same fixed, reproducible selection rule as notebook 00.
matches = first_n_matching(cuad, lambda ex: ex["question"] == "Governing Law", 2)
source_document = matches[1]["context"]

print(f"Source document: {len(source_document)} characters")
print(source_document[:300], "...")


## Generating candidates

One prompt, asking for several diverse questions at once -- covering different clause types, not five variations on the same one. Each candidate carries its own supporting quote, so the grounding check right after this doesn't have to guess where the answer came from.

*(This cell calls the live Groq API -- not yet run in this session; the account hit its daily token cap partway through building this series. Written and reasoned through carefully, but treat the exact output as unverified until it's actually run.)*


In [ ]:
GENERATION_PROMPT = """Given the following contract, generate 8 diverse question-answer pairs \
that could be used to test a RAG system's ability to answer questions about this document.

For each pair, provide:
- question: a natural question a user might actually ask
- answer: the answer, in your own words
- source_quote: the exact verbatim sentence or phrase from the document that supports the answer -- \
copy it exactly, do not paraphrase the quote itself

Cover different clause types across the document (parties involved, term/termination, liability, \
payment, services provided, governing law, etc) -- not multiple questions about the same clause.

Return strict JSON only, no other text:
{{"qa_pairs": [{{"question": "...", "answer": "...", "source_quote": "..."}}]}}

Document:
{document}"""


def generate_candidates(document: str) -> list[dict]:
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": GENERATION_PROMPT.format(document=document)}],
        temperature=0.4,  # some diversity across the 8 questions is wanted here
        reasoning_effort="low",
    )
    return json.loads(response.choices[0].message.content)["qa_pairs"]


candidates = generate_candidates(source_document)
print(f"Generated {len(candidates)} candidate question-answer pairs")
for c in candidates:
    print(f"  Q: {c['question']}")


## Filter 1 — grounding: does the quote actually exist?

Before judging whether a question is *good*, check something more basic and fully automatable: is the `source_quote` the model claims backs each answer actually present in the document, or did generation quietly invent one? This doesn't need another model call -- it's a direct string check against text you already have.


In [ ]:
def is_grounded(candidate: dict, document: str) -> bool:
    quote = candidate["source_quote"].strip()
    return quote in document


grounded_candidates = []
for c in candidates:
    grounded = is_grounded(c, source_document)
    status = "OK" if grounded else "UNGROUNDED -- quote not found verbatim"
    print(f"[{status}] {c['question']}")
    if grounded:
        grounded_candidates.append(c)

print(f"\n{len(grounded_candidates)}/{len(candidates)} candidates have a verifiable source quote")


If anything got dropped here, that's the generation step inventing a quote that doesn't literally exist in the document -- close paraphrasing of a real sentence, or an answer that's plausible but not actually traceable to specific text. Either way, an eval question you can't point at a real source for isn't one you can trust as ground truth.


## Filter 2 — deduplication: near-identical questions

Ask a model for "8 diverse questions" and it will sometimes hand back two different phrasings of the same underlying question. Use embeddings (the same tool from notebook 04 of the basics series, not another Groq call) to catch paraphrases a plain string comparison would miss entirely.

The threshold matters more than it looks like it should -- checked against a few hand-written pairs before picking one: genuine rewordings of the same question ("What law governs this agreement?" vs. "Under what governing law does this agreement operate?") scored 0.88-0.90 with this embedding model; a superficial word swap ("agreement" to "contract") scored 0.79; two genuinely different questions topped out around 0.66. **0.8** sits in the gap between "same question, reworded" and "different question that happens to share some vocabulary" -- for *this* embedding model, on *this kind* of short question text. Change either of those and the right number would likely move too.


In [ ]:
import numpy as np


def cosine_similarity(a, b) -> float:
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))


question_embeddings = embedder.encode([c["question"] for c in grounded_candidates])

DEDUP_THRESHOLD = 0.8

deduped = []
kept_indices = []
for i, c in enumerate(grounded_candidates):
    is_duplicate = any(
        cosine_similarity(question_embeddings[i], question_embeddings[j]) > DEDUP_THRESHOLD
        for j in kept_indices
    )
    if not is_duplicate:
        deduped.append(c)
        kept_indices.append(i)

print(f"{len(deduped)}/{len(grounded_candidates)} candidates survived deduplication")
for c in deduped:
    print(f"  Q: {c['question']}")


## Filter 3 — is this actually a good question?

Grounding and deduplication are mechanical checks -- useful, but they don't catch a question that's ambiguous, trivially guessable without the document, or answerable in a way that doesn't actually test anything interesting. That judgment is exactly the skill notebooks 01 through 06 built. Apply it here, on your own generated set, the same way.


In [ ]:
for i, c in enumerate(deduped):
    print(f"[{i}] Q: {c['question']}")
    print(f"    A: {c['answer']}")
    print(f"    Source: {c['source_quote'][:150]}")
    print()


In [ ]:
quality_review = {
    # index: "keep" | "drop", with a reason -- read each printed pair above before deciding
    # 0: {"verdict": "keep", "reason": ""},
}

print(f"Reviewed {len(quality_review)}/{len(deduped)}")


## Saving the final eval set

Whatever survives all three filters becomes a real, usable benchmark -- built from a document nobody had pre-labeled, the way most real eval sets actually get made.


In [ ]:
final_set = [c for i, c in enumerate(deduped) if quality_review.get(i, {}).get("verdict") == "keep"]

output_path = os.path.join(DATA_DIR, "synthetic_eval_set.json")
with open(output_path, "w") as f:
    json.dump(final_set, f, indent=2)

print(f"Saved {len(final_set)} reviewed question-answer pairs to {output_path}")


## Where human review is still necessary

Three filters, and only one of them (grounding) is fully mechanical. Deduplication needed an embedding model, and even that only catches near-duplicates, not two questions that are worded completely differently but test the exact same fact. The third filter -- is this actually a *good* question -- has no shortcut at all. This is the honest shape of synthetic dataset generation at any real scale: automation gets you from zero questions to a large candidate pool cheaply, and a human (or a very carefully calibrated judge, per notebook 07) still has to decide which candidates are worth keeping. Skipping that last step is how benchmarks quietly fill up with bad questions that nobody ever notices, because nobody ever read them.


## What we learned

**Terms to keep, in plain English:**

| Term | Plain meaning |
|---|---|
| Synthetic generation | Using a model to produce candidate eval questions from source documents, instead of authoring them by hand |
| Grounding check | Verifying a generated answer's supporting quote actually exists in the source, mechanically |
| Quality filtering | The review pass -- automated where possible, human where it can't be -- that decides which candidates become real ground truth |

The three-filter pattern here -- mechanical grounding check, embedding-based dedup, human judgment -- scales the same way at 8 candidates or 8,000: only the last step stays genuinely manual, and it's the one that matters most.

**Next up:** notebook `11` covers what a production evaluation pipeline logs on every real query, including cost, latency, and a first look at safety checks -- then notebook `12` closes the series by using everything built so far to answer, concretely, whether a real change to a pipeline helped.

**Exercises:**

- Finish `quality_review` for every surviving candidate, then actually run `own_faithfulness` (notebook 08) or a manual read against your final set -- does the "gold" answer you generated hold up the same way CUAD's did?
- Lower `DEDUP_THRESHOLD` from `0.8` to `0.65` and see what gets merged that shouldn't -- topically related questions about different clauses can score surprisingly close to that range, which is exactly the false-merge risk a too-loose threshold creates.
- Try the same generation prompt on a second CUAD contract and compare: does the model reliably cover different clause types, or does it gravitate toward the same one or two every time?
